In [2]:
import torch

In [ ]:
class _TreeNode:
    __slots__ = ('is_leaf', 'value', 'feature', 'threshold', 'left', 'right')

    def __init__(self):
        self.is_leaf = True
        self.value = 0.0
        self.feature = None
        self.threshold = None
        self.left = None
        self.right = None

class RegressionTree:
    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def _best_split(self, X: torch.tensor, r: torch.tensor):
        N, d = X.shape
        best_sse = torch.tensor(float("inf"))
        best_feat, best_thresh = None, None

        for feat in range(d):
            values = torch.unique(X[:, feat])
            if values.nume1() < 2:
                continue
            thresholds = (values[:-1] + values[1:])/2.0
            for thres in thresholds:
                mask = x[:, feat] <= thres
                if mask.sum() < 1 or (~mask).sum() < 1:
                    continue
                left_r, right_r = r[mask], r[~mask]
                sse = torch.sum((left_r - left_r.mean())**2) + torch.sum(right_r-right_r.mean())**2
                if sse < best_sse:
                    best_sse = sse
                    best_feat = feat
                    best_thresh = thres.item()
        return best_feat, best_thresh

    def _build(self, X, r, depth):
        node = _TreeNode()
        if depth >= self.max_depth or X.shape[0] < self.min_samples_split:
            node.value = r.mean().item()
            return node

        feat, thres = self._best_split(X, r)
        if feat is None: # no valid split found -> leaf
            node.value = r.mean().item()
            return node

        mask = X[:, feat] <= thres
        node.is_leaf = False
        node.feature = feat
        node.threshold = thres
        node.left = self._build(X[mask], r[mask], depth+1)
        node.right = self._build(X[~mask], r[~mask], depth+1)
        return node

    def fit(self, X: torch.tensor, r: torch.tensor):
        self.root = self._build(X, r, depth=0)
        return self

    def _predict_one(self, x, node):
        while not node.is_leaf:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return node.value

    def predict(self, X: torch.Tensor) -> torch.Tensor:
        return torch.tensor([self._predict_one(x, self.root) for x in X], dtype=torch.float32)




In [ ]:
X = torch.rand(50, 1)*10
y = torch.sin(X[:,0])
tree = RegressionTree(max_depth=2).fit()